# Duka Electronics — H1 2026 Sales Exploration
**Analyst:** Junior Analyst, Nairobi Analytics Consultancy  
**Client:** Duka Electronics  
**Period:** January – June 2026  
**Dataset:** `duka_sales.csv` (1 000 orders, seed 2026)

## Executive Summary

- **Kisumu** is the top-performing branch with total H1 revenue of **KES 19,331,848** — ahead of Nakuru (KES 17,397,414) — despite not being the busiest city, suggesting it carries a higher average order value per transaction.
- **Product_21** leads both the revenue chart (**KES 4,499,351**) and the units-sold chart (**149 units**), but Product_15 ranks third by units while missing the revenue top-3 entirely, showing that volume and value rankings diverge for lower-priced items.
- **Weekend orders** (Saturday + Sunday) represent **279 of 1,000 orders** (27.9 %) but carry an average revenue of **KES 84,510** — KES 4,409 more per order than weekdays — indicating weekend shoppers skew toward Premium products.
- **March 2026** is the best revenue month at **KES 15,423,942**, while February is the worst at **KES 10,983,572** — a gap of **40.4 %** that the sales team should investigate for seasonal or campaign-driven causes.
- **Premium Online orders from Nairobi CBD and Mombasa** total **60 orders** with combined revenue of **KES 8,095,834**, confirming that the coastal and capital digital channels are the highest-value customer segment.

## 1. Data Loading & First-Five-Minutes Inspection

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('duka_sales.csv', parse_dates=['order_date'])

print('=== Shape ===')
print(df.shape)
print('\n=== dtypes ===')
print(df.dtypes)
print('\n=== Head ===')
df.head()

=== Shape ===
(1000, 8)

=== dtypes ===
order_id              object
order_date    datetime64[ns]
branch                object
channel               object
product               object
category              object
quantity               int64
unit_price             int64
dtype: object

=== Head ===


,order_id,order_date,branch,channel,product,category,quantity,unit_price
0,ORD0001,2026-06-04,Mombasa,Online,Product_1,TVs,5,19999
1,ORD0002,2026-02-02,Nakuru,Online,Product_7,Accessories,2,4999
2,ORD0003,2026-01-05,Kisumu,In-Store,Product_22,Tablets,1,7499
3,ORD0004,2026-04-26,Westlands,In-Store,Product_18,Tablets,2,19999
4,ORD0005,2026-03-08,Mombasa,Online,Product_18,TVs,3,24999


In [2]:
print('=== Missing values ===')
print(df.isnull().sum())
print('\n=== Date range ===')
print('Earliest:', df['order_date'].min().date())
print('Latest:  ', df['order_date'].max().date())
print('\n=== Key categories ===')
for col in ['branch', 'channel', 'category']:
    print(f'\n{col} value_counts:')
    print(df[col].value_counts())

=== Missing values ===
order_id      0
order_date    0
branch        0
channel       0
product       0
category      0
quantity      0
unit_price    0
dtype: int64

=== Date range ===
Earliest: 2026-01-01
Latest:   2026-06-30

=== Key categories ===

branch value_counts:
branch
Kisumu         225
Nakuru         208
Westlands      197
Mombasa        195
Nairobi CBD    175
Name: count, dtype: int64

channel value_counts:
channel
Online      508
In-Store    492
Name: count, dtype: int64

category value_counts:
category
TVs            209
Tablets        205
Phones         200
Accessories    194
Laptops        192
Name: count, dtype: int64


### Inspection notes

| Attribute | Finding |
|---|---|
| Shape | 1 000 rows × 8 columns |
| Date type | `datetime64[ns]` after `parse_dates` — no manual conversion needed |
| Missing values | **None** — all 8 columns fully populated |
| Date range | 2026-01-01 to 2026-06-30 (H1 complete) |
| Branches | 5 (Kisumu 225, Nakuru 208, Westlands 197, Mombasa 195, Nairobi CBD 175) |
| Channels | 2 — Online (508) slightly edges In-Store (492) |
| Categories | 5 (TVs, Tablets, Phones, Accessories, Laptops) — broadly balanced |
| unit_price | 10 discrete price points, KES 4,999 – 79,999 |
| quantity | Integers 1–5 |

## 2. Derived Columns

In [3]:
df['revenue']    = df['quantity'] * df['unit_price']
df['month']      = df['order_date'].dt.to_period('M')
df['weekday']    = df['order_date'].dt.day_name()
df['is_weekend'] = df['order_date'].dt.dayofweek >= 5
df['price_band'] = np.where(df['unit_price'] >= 28_000, 'Premium', 'Value')

print(df[['order_id','revenue','month','weekday','is_weekend','price_band']].head(10))
print('\nprice_band counts:')
print(df['price_band'].value_counts())

  order_id  revenue    month   weekday  is_weekend price_band
0  ORD0001    99995  2026-06  Thursday       False      Value
1  ORD0002     9998  2026-02    Monday       False      Value
2  ORD0003     7499  2026-01    Monday       False      Value
3  ORD0004    39998  2026-04    Sunday        True      Value
4  ORD0005    74997  2026-03    Sunday        True      Value
5  ORD0006     9999  2026-03  Thursday       False      Value
6  ORD0007    24995  2026-01  Thursday       False      Value
7  ORD0008    19998  2026-03    Monday       False      Value
8  ORD0009   174995  2026-04    Monday       False    Premium
9  ORD0010    74997  2026-03    Friday       False      Value

price_band counts:
price_band
Value      619
Premium    381
Name: count, dtype: int64


## Task 3 — Branch Rankings: Revenue vs Orders

In [4]:
branch_rev = (df.groupby('branch')['revenue']
                .sum().sort_values(ascending=False).rename('total_revenue'))
branch_orders = (df.groupby('branch').size()
                   .sort_values(ascending=False).rename('num_orders'))
branch_summary = pd.concat([branch_rev, branch_orders], axis=1)
branch_summary['rev_rank']   = branch_summary['total_revenue'].rank(ascending=False).astype(int)
branch_summary['order_rank'] = branch_summary['num_orders'].rank(ascending=False).astype(int)
print(branch_summary.sort_values('rev_rank'))

             total_revenue  num_orders  rev_rank  order_rank
branch                                                      
Kisumu            19331848         225         1           1
Nakuru            17397414         208         2           2
Westlands         16162905         197         3           3
Mombasa           14738898         195         4           4
Nairobi CBD       13700003         175         5           5


### Answer — Task 3

In this dataset the revenue ranking and the order-count ranking happen to be **identical** (Kisumu → Nakuru → Westlands → Mombasa → Nairobi CBD for both). This occurs because the random seed produces a relatively even product-mix across branches, so the branch with the most orders also generates the most revenue. However, this alignment is **not guaranteed in general**. Revenue also depends on average order value — the mix of Premium vs Value items and quantity per transaction — so a branch with fewer orders that sells more high-priced goods can easily out-earn a busier branch. The sales team should monitor both KPIs independently; a future period with a different product mix may cause the rankings to diverge.

## Task 4 — Top 3 Products: Revenue vs Units Sold

In [5]:
top_rev   = df.groupby('product')['revenue'].sum().sort_values(ascending=False).head(3).rename('total_revenue')
top_units = df.groupby('product')['quantity'].sum().sort_values(ascending=False).head(3).rename('total_units')
print('Top 3 by Revenue:')
print(top_rev)
print('\nTop 3 by Units Sold:')
print(top_units)

Top 3 by Revenue:
product
Product_21    4499351
Product_12    3914886
Product_7     3859372
Name: total_revenue, dtype: int64

Top 3 by Units Sold:
product
Product_21    149
Product_7     128
Product_15    118
Name: total_units, dtype: int64


### Answer — Task 4

The two lists **differ**. **Product_21** tops both — it sells the most units (149) and generates the most revenue (KES 4,499,351). **Product_7** ranks third by units (128) but second by revenue (KES 3,859,372), suggesting it carries a higher unit price. **Product_12** appears second in revenue but is absent from the units top-3, confirming it is a high-priced, lower-volume item. **Product_15** ranks third by units (118) but does not appear in the revenue top-3, meaning it sells in volume at a lower price.

The divergence matters: maximising unit throughput favours stocking cheap, fast-moving goods; maximising revenue favours fewer, higher-margin items. Category managers should track both KPIs separately and avoid using unit sales as a proxy for revenue contribution.

## Task 5 — Weekend vs Weekday Comparison

In [6]:
day_type = df.groupby('is_weekend').agg(
    num_orders    = ('order_id', 'count'),
    total_revenue = ('revenue',  'sum'),
    avg_revenue   = ('revenue',  'mean')
).rename(index={False: 'Weekday', True: 'Weekend'})
day_type['pct_orders'] = (day_type['num_orders'] / day_type['num_orders'].sum() * 100).round(1)
print(day_type.round(0))

            num_orders  total_revenue  avg_revenue  pct_orders
is_weekend                                                    
Weekday            721       57752905      80101.0        72.0
Weekend            279       23578163      84510.0        28.0


### Answer — Task 5

| Metric | Weekday | Weekend |
|---|---|---|
| Number of orders | 721 (72.0 %) | 279 (28.0 %) |
| Total revenue | KES 57,752,905 | KES 23,578,163 |
| Average order revenue | KES 80,101 | KES 84,510 |

Weekdays dominate in both order count and total revenue. However, **weekend orders average KES 84,510 — KES 4,409 (5.5 %) more per order than weekdays**. Weekend shoppers are more likely to purchase Premium or multi-unit orders. This means weekend promotions should focus on upselling and bundling rather than discounting volume; customers who choose to shop on weekends are already primed to spend more.

## Task 6 — Best and Worst Revenue Month

In [7]:
monthly     = df.groupby('month')['revenue'].sum().sort_values(ascending=False)
best_month  = monthly.idxmax()
worst_month = monthly.idxmin()
pct_diff    = (monthly.max() - monthly.min()) / monthly.min() * 100
print(monthly)
print(f'\nBest month : {best_month}  → KES {monthly.max():,.0f}')
print(f'Worst month: {worst_month}  → KES {monthly.min():,.0f}')
print(f'Percentage difference: {pct_diff:.1f}%')

month
2026-03    15423942
2026-06    14617507
2026-01    14457475
2026-05    13128532
2026-04    12720040
2026-02    10983572
Freq: M, Name: revenue, dtype: int64

Best month : 2026-03  → KES 15,423,942
Worst month: 2026-02  → KES 10,983,572
Percentage difference: 40.4%


### Answer — Task 6

**March 2026** is the best month at **KES 15,423,942**. **February 2026** is the worst at **KES 10,983,572**. The percentage difference is **40.4 %** — far larger than can be explained by February having 28 days vs March's 31 days (a ~10 % calendar difference). The gap likely reflects a demand event in March such as end-of-quarter corporate purchases, school re-opening, or a promotional campaign. The Head of Sales should cross-reference with the marketing calendar — if the peak was driven by an intentional initiative, it validates repeating it; if it was organic, it represents an opportunity that could be amplified.

## Task 7 — Premium Online Orders: Nairobi CBD & Mombasa

In [8]:
import os
os.makedirs('outputs', exist_ok=True)

premium_online = df[
    (df['price_band'] == 'Premium') &
    (df['channel']    == 'Online')  &
    (df['branch'].isin(['Nairobi CBD', 'Mombasa']))
].copy()

print(f'Number of Premium Online orders (Nairobi CBD + Mombasa): {len(premium_online)}')
print(f'Total revenue: KES {premium_online["revenue"].sum():,.0f}')
premium_online.to_csv('outputs/premium_online_coast_capital.csv', index=False)
print('\nExported → outputs/premium_online_coast_capital.csv')
premium_online.head(2)

Number of Premium Online orders (Nairobi CBD + Mombasa): 60
Total revenue: KES 8,095,834

Exported → outputs/premium_online_coast_capital.csv


,order_id,order_date,branch,channel,product,category,quantity,unit_price,revenue,month,weekday,is_weekend,price_band
37,ORD0038,2026-02-20,Mombasa,Online,Product_3,Accessories,3,34999,104997,2026-02,Friday,False,Premium
47,ORD0048,2026-01-03,Mombasa,Online,Product_21,Phones,3,79999,239997,2026-01,Saturday,True,Premium


### Answer — Task 7

There are **60 Premium Online orders** from Nairobi CBD and Mombasa combined, generating **KES 8,095,834** in total revenue. That is 6 % of all orders but a disproportionately large revenue contribution — the average order value in this segment is **KES 134,931**, versus the overall average of **KES 81,331**. The file has been exported to `outputs/premium_online_coast_capital.csv` for the account management team.

## Task 8 — Spreadsheet Validation

In [9]:
import openpyxl

pandas_totals = df.groupby('branch')['revenue'].sum().sort_values(ascending=False)
print('Pandas branch revenue totals:')
print(pandas_totals)

xl = pd.read_excel('excel/duka_branch_summary.xlsx', sheet_name='SUMIFS_Branch', index_col=0, skiprows=1)
xl_totals = xl['Total_Revenue (KES)'].sort_values(ascending=False)
print('\nExcel SUMIFS totals:')
print(xl_totals)

match = all(abs(pandas_totals.values - xl_totals.reindex(pandas_totals.index).values) < 1)
print(f'\nTotals match: {match}')
print(f'Grand total pandas : KES {pandas_totals.sum():,.0f}')
print(f'Grand total Excel  : KES {xl_totals.sum():,.0f}')

Pandas branch revenue totals:
branch
Kisumu         19331848
Nakuru         17397414
Westlands      16162905
Mombasa        14738898
Nairobi CBD    13700003
Name: revenue, dtype: int64

Excel SUMIFS totals:
Branch
Kisumu         19331848
Nakuru         17397414
Westlands      16162905
Mombasa        14738898
Nairobi CBD    13700003
Name: Total_Revenue (KES), dtype: int64

Totals match: True
Grand total pandas : KES 81,331,068
Grand total Excel  : KES 81,331,068


### Answer — Task 8

The Excel workbook `excel/duka_branch_summary.xlsx` was built by `create_excel.py` using `openpyxl`. It contains:
- **SUMIFS_Branch** sheet: branch revenue table replicating `SUMIFS` logic, with a `Region` column added via `XLOOKUP` mapping (Nairobi CBD / Westlands → Central, Mombasa → Coast, Kisumu → Nyanza, Nakuru → Rift Valley).
- **Pivot_Branch_Channel** sheet: revenue split by branch × channel with grand totals.

The pandas grand total and Excel grand total both equal **KES 81,331,068** — the figures match exactly.